## Agentic AI Research Agent LangGraph + OpenAI + Wikipedia
This notebook builds a small ReAct-style agent that can decide on its own whether it needs to look something up on Wikipedia or just answer directly.

### Install dependencies

In [1]:
!pip install --upgrade langchain langchain-community langgraph openai langchain_openai wikipedia

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=55b9258cb18e69b9af5d846048bf29e22838c2da7fcb0bb84e266cf469f8870e
  Stored in directory: /root/.cache/pip/wheels/79/1d/c8/b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
Successfully built wikipedia
  Attempting uninstall: requests
    F

### Set up the Wikipedia tool
This gives the agent the ability to search Wikipedia when it needs factual information.

In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

/tmp/ipykernel_5983/3243019729.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [3]:
wiki_tool.run({"query": "AI agents"})

'Page: AI agent\nSummary: An AI agent, also known as agentic AI in buzzword form, is an artificial intelligence program that can pursue goals, use software or other tools, and take actions with some level of autonomy. Agentic AI contrasts with tool-like AI use for narrow, specific tasks such as answer'

### Set up the LLM (the agent's "brain")

In [6]:
!pip install -U langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.8 MB/s eta 0:00:00


In [12]:
from langchain_groq import ChatGroq
llm = ChatGroq(temperature=0, api_key="prajw*********key", model="openai/gpt-oss-120b")

### Bind tools to the LLM
This tells the model which tools it is allowed to call, and lets it decide on its own when to use them.

In [13]:
tools = [wiki_tool]
llm_with_tools = llm.bind_tools(tools)
result = llm_with_tools.invoke("Hello world!")
result
result.content

'Hello! How can I assist you today?'

### Build the agent using LangGraph's prebuilt ReAct constructor
`create_react_agent` wires up the LLM + tools into a "reason → act (call tool) → observe → repeat" loop automatically, so we don't have to build the StateGraph nodes/edges by hand.

In [14]:
from langgraph.prebuilt import create_react_agent
agent_executor = create_react_agent(llm, tools)

/tmp/ipykernel_5983/2525888991.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


### Test 1 — a question that does NOT need a tool

In [15]:
from langchain_core.messages import HumanMessage
response = agent_executor.invoke({"messages": [HumanMessage(content="hi!")]})
response["messages"]

[HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='ec425406-3861-41ff-a8f0-b6e9fd5b4423'),
 AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user just says "hi!". We should respond politely. No need for tools.'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 158, 'total_tokens': 194, 'completion_time': 0.074376078, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.006315661, 'prompt_tokens_details': None, 'queue_time': 0.114777438, 'total_time': 0.080691739}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07695-e35c-7c80-8195-5666ff6e2382-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'output_tokens': 36, 'total_tokens': 194, 'output_token_details': {'reasoning': 18}})]

In [16]:
# Print only the final assistant reply (just the answer text, no metadata)
print(response["messages"][-1].content)

Hello! How can I help you today?


### Test 2 — a question that SHOULD trigger the Wikipedia tool

In [17]:
# Ask a factual question. This should make the agent decide to call the "wikipedia" tool.
response = agent_executor.invoke({"messages": [HumanMessage(content="what is agentic ai")]})

# This time the messages list will contain 4 items:
# 1. HumanMessage      - the original question
# 2. AIMessage         - the model deciding to call the "wikipedia" tool (tool_calls populated)
# 3. ToolMessage       - the actual Wikipedia result returned back into the conversation
# 4. AIMessage (final) - the model's final answer, written using the Wikipedia result
response["messages"]

[HumanMessage(content='what is agentic ai', additional_kwargs={}, response_metadata={}, id='635c3ef0-d9c6-4039-b841-5bc944c41883'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "what is agentic AI". Need to explain concept. Provide definition, context, examples, implications, maybe references. Could cite Wikipedia. Let\'s search.', 'tool_calls': [{'id': 'fc_b9914500-e2f6-4e13-8d4e-cb13365e752d', 'function': {'arguments': '{"query":"Agentic AI"}', 'name': 'wikipedia'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 161, 'total_tokens': 225, 'completion_time': 0.133103276, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.006426037, 'prompt_tokens_details': None, 'queue_time': 0.114721457, 'total_time': 0.139529313}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provi

In [18]:
# Print only the final, human-readable answer generated after using the Wikipedia tool
print(response["messages"][-1].content)

**Agentic AI** (often called an *AI agent*) refers to an artificial‑intelligence system that can **pursue its own goals, select and use tools, and take actions autonomously**—rather than simply responding to a single, narrowly defined prompt or performing a single, pre‑programmed task.

Below is a concise overview of what makes an AI “agentic,” how it differs from more “tool‑like” AI, and why it matters.

---

## 1. Core Definition
- **AI Agent / Agentic AI** – An AI program that can **decide what to do, plan how to do it, and execute actions** in an environment (digital or physical) to achieve a goal it has been given or has inferred.  
- The term is a **buzz‑word version** of “AI agent” used in academic literature and industry discussions.

---

## 2. Key Characteristics

| Characteristic | What It Means | Example |
|----------------|---------------|---------|
| **Goal‑oriented** | The system has a *desired outcome* (explicitly programmed or inferred) and works toward it. | A persona

In [19]:
response = agent_executor.invoke({"messages": [HumanMessage(content="what is gen ai")]})
response["messages"]
print(response["messages"][-1].content)

**Generative AI (Gen AI)** is a branch of artificial intelligence that creates new content—such as text, images, audio, video, code, or even 3‑D models—rather than just analyzing or classifying existing data.  

---

## Core Idea
- **Model + Training Data → New Output**  
  A generative model learns the statistical patterns of a large dataset (e.g., millions of sentences, photos, or songs). Once trained, it can sample from that learned distribution to produce original artifacts that resemble the training material but are not direct copies.

---

## Main Types of Generative Models

| Model family | Typical architecture | What it’s good at |
|--------------|----------------------|-------------------|
| **Transformer‑based language models** (e.g., GPT‑4, LLaMA, Claude) | Decoder‑only or encoder‑decoder transformers | Coherent, context‑aware text, code, chat, summarization |
| **Diffusion models** (e.g., Stable Diffusion, DALL‑E 3, Imagen) | Latent diffusion + UNet denoising | High‑quality